## Project 2
In this project, I explore how economic development and internet adoption move together over time.  
More specifically, I look at the relationship between GDP per capita and Internet usage in different countries.
My goal is not to prove causality, but to see whether higher income levels tend to go hand in hand with higher internet penetration.

### Data sources

GDP per capita (current US$)
  World Bank, World Development Indicators,
  Dataset page: https://data.worldbank.org/indicator/NY.GDP.PCAP.CD  

Individuals using the Internet (% of population)
  World Bank, World Development Indicators
  Dataset page: https://data.worldbank.org/indicator/IT.NET.USER.ZS


## Loading the data

First I import the libraries I need and read the two CSV files into pandas DataFrames.

The World Bank CSVs include four lines of metadata at the top, so I use `skiprows=4` to skip those rows and start reading from the actual header row with the column names.

In [ ]:
import pandas as pd
import plotly.express as px

In [ ]:
df1 = pd.read_csv("user.csv", skiprows=4)
df2 = pd.read_csv("GDP.csv", skiprows=4)

## check dataframes
Then I use .head to check dataframes, including columns' name.

In [ ]:
df1.head()

In [ ]:
df2.head()

## Reshaping the data to long format

In the raw World Bank files, each year is stored as a separate column (1960, 1961, …, 2024).  
This “wide” format is not convenient for analysis or plotting, so I reshape both datasets into a “long” format.

Because not all columns are digits, I first identify all the year columns by selecting the columns whose names are digits.  
Then I use pd.melt to turn those year columns into a single Year column and a single value column.


In [ ]:
year_cols1 = [col for col in df1.columns if col.isdigit()]
year_cols2 = [col for col in df2.columns if col.isdigit()]

df_long1 = df1.melt(
    id_vars=["Country Name", "Country Code"],
    value_vars=year_cols1,
    var_name="Year",
    value_name="Value",
)
df_long2 = df2.melt(
    id_vars=["Country Name", "Country Code"],
    value_vars=year_cols2,
    var_name="Year",
    value_name="Value",
)

## Handling missing values

Many country–year combinations have no data, especially in the earlier decades.  
To keep the analysis simple and make the plots easier to read, I drop rows where the value is missing in either dataset.

In [ ]:
df_long1 = df_long1.dropna(subset=["Value"])
df_long2 = df_long2.dropna(subset=["Value"])

## Focusing on two countries

To make the relationship easier to see, I focus on two countries:  
the United States and China, which are all super balls in the world.

Here I filter both long-format datasets to keep only these two countries.

In [ ]:
Countries = ["United States", "China"]

In [ ]:
filt_df1 = df_long1["Country Name"].isin(Countries)
filt_df2 = df_long2["Country Name"].isin(Countries)

In [ ]:
research_df1 = df_long1[filt_df1]
research_df2 = df_long2[filt_df2]

## check dataframes which are used to make charts.

In [ ]:
research_df1.head()

# Merging GDP and Internet usage

Next, I rename the value columns to something more descriptive:  
Internet Usage for the first dataset and GDP for the second one.

Then I merge the two datasets on Country Name, Country Code, and Year.  
After this merge, each row represents a country–year pair with both GDP per capita and Internet usage.

In [ ]:
research_df1 = research_df1.rename(columns={"Value": "Internet Usage"})
research_df2 = research_df2.rename(columns={"Value": "GDP"})

In [ ]:
merged = pd.merge(
    research_df1, research_df2, on=["Country Name", "Country Code", "Year"], how="inner"
)
merged.head()

## Visualizing the relationship: China

For China, I create a scatter plot where each point is one year.  
The x-axis shows GDP per capita, and the y-axis shows Internet usage (% of the population).  
This lets me see how the two indicators move together over time.

In [ ]:
country = "China"
plot_df = merged[merged["Country Name"] == country]

fig = px.scatter(
    plot_df,
    x="GDP",
    y="Internet Usage",
    hover_data=["Year"],
    title=f"{country}: GDP vs Internet usage",
)
fig.show()


## Visualizing the relationship: United States

I repeat the same plot for the United States.  
Again, each point represents a single year, with GDP per capita on the x-axis and Internet usage on the y-axis.

In [ ]:
country = "United States"
plot_df = merged[merged["Country Name"] == country]

fig = px.scatter(
    plot_df,
    x="GDP",
    y="Internet Usage",
    hover_data=["Year"],
    title=f"{country}: GDP vs Internet usage",
)
fig.show()

## Takeaways

Looking at China and the United States, the pattern is pretty clear: as GDP per capita goes up, Internet usage also rises. In China, Internet adoption takes off once GDP starts growing quickly, and it reaches very high levels as income improves. In the U.S., Internet usage climbs fast at first and then levels off once most people are already online.

Overall, the chart suggests a positive relationship between economic development and how widely people use the Internet. It doesn’t prove one causes the other, but the trend shows that higher income levels usually come with higher digital access.